# Data Quality Project — Colab Runner

**Đề tài:** Xây dựng khung đánh giá chất lượng dữ liệu quy mô lớn có nhận biết ngữ cảnh, gắn với hiệu năng mô hình downstream

Notebook này dùng để thiết lập môi trường chạy thí nghiệm trên Google Colab (mount Drive, cài thư viện, tạo cấu trúc thư mục đồng bộ với bản local trên máy Windows). Tham khảo `docs/checklist-trien-khai.md` để theo dõi tiến độ, và `docs/pipeline-thi-nghiem.md` / `docs/kien-truc-mo-hinh-va-ky-thuat.md` cho chi tiết kỹ thuật từng module.

**Lưu ý:** Notebook chỉ thiết lập môi trường — các bước tải dataset thật, cài thêm thư viện, hay chạy thí nghiệm full đều cần xác nhận riêng trước khi thực thi, theo đúng quy tắc đã thống nhất.

## 1. Mount Google Drive

Dùng Drive để lưu dữ liệu, code, kết quả — không bị mất khi hết phiên Colab (ổ đĩa tạm của Colab sẽ reset sau mỗi phiên).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Thiết lập cấu trúc thư mục project trên Drive

Tạo (nếu chưa có) cấu trúc thư mục giống hệt bản local trên máy Windows, xem `docs/cau-truc-thu-muc-project.md`.

In [2]:
import os

PROJECT_DIR = '/content/drive/MyDrive/data-quality-project'

SUBDIRS = [
    'configs',
    'data/raw/beijing_air_quality',
    'data/raw/higgs_sample',
    'data/raw/higgs_full',
    'data/raw/covertype',
    'data/processed',
    'src/profiling',
    'src/pollution',
    'src/downstream',
    'src/experiment_runner',
    'src/metamodel',
    'src/baselines',
    'src/scalability',
    'notebooks',
    'results/figures',
    'tests',
    'docs',
]

for sub in SUBDIRS:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

print("Project dir:", PROJECT_DIR)
print("Đã tạo đủ", len(SUBDIRS), "thư mục con (nếu chưa có).")

Project dir: /content/drive/MyDrive/data-quality-project
Đã tạo đủ 17 thư mục con (nếu chưa có).


**Đồng bộ code/docs:** thư mục trên vẫn trống. Có 2 cách đưa code/docs từ máy Windows lên đây:

- **Cách thủ công (đơn giản nhất lúc này):** mở Google Drive trên trình duyệt, kéo-thả toàn bộ nội dung `data-quality-project/` từ máy Windows vào đúng thư mục `MyDrive/data-quality-project/`.
- **Cách qua Git (khi đã có repo):** nếu sau này đưa project lên GitHub, dùng `!git clone <url> {PROJECT_DIR}` thay cho việc tạo thư mục rỗng ở trên.

Hiện tại `src/`, `docs/`, `configs/` chưa có file thật (đúng trạng thái Giai đoạn 0 vừa hoàn tất) — chỉ cần đồng bộ khi bắt đầu có code/dataset.

## 3. Cài đặt thư viện

Trên Colab bật luôn `pyspark`, `shap`, `xgboost` (đang để comment trong `requirements.txt` ở bản local vì máy Windows cần tiết kiệm dung lượng).

In [3]:
requirements_colab = """pandas>=2.0
numpy>=1.24
matplotlib>=3.7
scikit-learn>=1.3
tensorflow>=2.15
pyspark>=3.5
shap>=0.44
xgboost>=2.0
"""

req_path = os.path.join(PROJECT_DIR, 'requirements.txt')
with open(req_path, 'w') as f:
    f.write(requirements_colab)

!pip install -q -r {req_path}
print("Đã cài xong thư viện theo", req_path)

Đã cài xong thư viện theo /content/drive/MyDrive/data-quality-project/requirements.txt


## 4. Kiểm tra môi trường

In [4]:
import sys
print("Python:", sys.version.split()[0])

import pandas as pd, numpy as np, sklearn
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

import shap
print("shap:", shap.__version__)

import xgboost
print("xgboost:", xgboost.__version__)

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("data-quality-project").getOrCreate()
print("pyspark:", spark.version, "(local[*] mode — xem docs/kien-truc-mo-hinh-va-ky-thuat.md mục 6)")
spark.stop()

Python: 3.13.16
pandas: 2.2.3
numpy: 2.1.3
scikit-learn: 1.6.1
shap: 0.52.0
xgboost: 3.4.1
pyspark: 4.0.4 (local[*] mode — xem docs/kien-truc-mo-hinh-va-ky-thuat.md mục 6)


## 5. Thêm `src/` vào `sys.path`

Để import các module khi đã viết code, ví dụ `from profiling import pandas_profiler`.

In [5]:
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Đã thêm vào sys.path:", SRC_DIR)

Đã thêm vào sys.path: /content/drive/MyDrive/data-quality-project/src


## 6. Load configs

Chạy sau khi đã tạo `seeds.yaml`, `pollution_levels.yaml`, `experiment_matrix.yaml` trong `configs/` (việc của Giai đoạn 1, xem checklist).

In [6]:
import yaml

def load_config(name):
    path = os.path.join(PROJECT_DIR, 'configs', name)
    if not os.path.exists(path):
        print(f"[Chưa có] {name} — cần tạo ở Giai đoạn 1 (xem docs/checklist-trien-khai.md)")
        return None
    with open(path) as f:
        return yaml.safe_load(f)

seeds = load_config('seeds.yaml')
pollution_levels = load_config('pollution_levels.yaml')
experiment_matrix = load_config('experiment_matrix.yaml')

## 7. Tải dataset (Giai đoạn 1 — nhóm B và C)

Chạy từng cell theo thứ tự 7.1 → 7.2 → 7.3. Dữ liệu thô (`data/raw/`) giữ nguyên bản gốc, **không sửa**. Các cell đều bỏ qua nếu dữ liệu đã có trên Drive, nên chạy lại không tải lặp.

**7.1 Beijing** (~vài MB) · **7.2 Covertype** (~75MB) · **7.3 HIGGS mẫu** (tải ~2.8GB về ổ tạm của Colab, chỉ lưu mẫu lên Drive).

In [7]:
# --- 7.1 Beijing Multi-Site Air Quality (UCI, 12 trạm) ---
import glob, zipfile, shutil, urllib.request

BEIJING_DIR = os.path.join(PROJECT_DIR, 'data/raw/beijing_air_quality')
if len(glob.glob(os.path.join(BEIJING_DIR, '**/*.csv'), recursive=True)) >= 12:
    print("Beijing đã có trên Drive, bỏ qua.")
else:
    zip_path = '/content/beijing.zip'
    urllib.request.urlretrieve(
        'https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip', zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content/beijing_tmp')
    # file zip có thể lồng thêm 1 zip bên trong
    for inner in glob.glob('/content/beijing_tmp/**/*.zip', recursive=True):
        with zipfile.ZipFile(inner) as z:
            z.extractall('/content/beijing_tmp')
    csvs = sorted(glob.glob('/content/beijing_tmp/**/PRSA_Data_*.csv', recursive=True))
    print("Số file CSV trạm tìm thấy:", len(csvs))
    assert len(csvs) == 12, "Kỳ vọng 12 file trạm — kiểm tra lại cấu trúc zip"
    for p in csvs:
        shutil.copy(p, BEIJING_DIR)

files = sorted(glob.glob(os.path.join(BEIJING_DIR, 'PRSA_Data_*.csv')))
print(len(files), "file trong", BEIJING_DIR)
_df = pd.read_csv(files[0])
print(_df.shape); print(_df.head(3))

Beijing đã có trên Drive, bỏ qua.
12 file trong /content/drive/MyDrive/data-quality-project/data/raw/beijing_air_quality
(35064, 18)
   No  year  month  day  hour  PM2.5  PM10  SO2   NO2     CO    O3  TEMP  \
0   1  2013      3    1     0    4.0   4.0  4.0   7.0  300.0  77.0  -0.7   
1   2  2013      3    1     1    8.0   8.0  4.0   7.0  300.0  77.0  -1.1   
2   3  2013      3    1     2    7.0   7.0  5.0  10.0  300.0  73.0  -1.1   

     PRES  DEWP  RAIN   wd  WSPM       station  
0  1023.0 -18.8   0.0  NNW   4.4  Aotizhongxin  
1  1023.2 -18.2   0.0    N   4.7  Aotizhongxin  
2  1023.5 -18.2   0.0  NNW   5.6  Aotizhongxin  


In [8]:
# --- 7.2 Covertype (full 581.012 dòng, qua sklearn) ---
COVER_DIR = os.path.join(PROJECT_DIR, 'data/raw/covertype')
cover_path = os.path.join(COVER_DIR, 'covertype_raw.parquet')

if os.path.exists(cover_path):
    print("Covertype đã có trên Drive, bỏ qua.")
else:
    from sklearn.datasets import fetch_covtype
    cov = fetch_covtype(as_frame=True, data_home='/content/sk_data')
    cover_df = cov.frame            # 54 feature + cột target 'Cover_Type' (1-7)
    cover_df.to_parquet(cover_path, index=False)
    print("Đã lưu", cover_path)

_cov = pd.read_parquet(cover_path)
print(_cov.shape)   # kỳ vọng (581012, 55)
print(_cov['Cover_Type'].value_counts().sort_index())

Covertype đã có trên Drive, bỏ qua.
(581012, 55)
Cover_Type
1    211840
2    283301
3     35754
4      2747
5      9493
6     17367
7     20510
Name: count, dtype: int64


In [9]:
# --- 7.3 HIGGS mẫu (stratified 1.000.000 dòng từ 11M) ---
# Tải file ~2.8GB về ổ tạm /content (KHÔNG lưu lên Drive), chỉ lưu mẫu lên Drive.
HIGGS_DIR = os.path.join(PROJECT_DIR, 'data/raw/higgs_sample')
higgs_path = os.path.join(HIGGS_DIR, 'higgs_sample_raw.parquet')
N_SAMPLE = 1_000_000

if os.path.exists(higgs_path):
    print("HIGGS mẫu đã có trên Drive, bỏ qua.")
else:
    assert seeds is not None, "Cần chạy cell load configs (mục 6) trước"
    seed = seeds['base_seed']
    gz = '/content/HIGGS.csv.gz'
    if not os.path.exists(gz):
        !wget -q -O {gz} https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz
    cols = ['target'] + [f'f{i:02d}' for i in range(1, 29)]
    full = pd.read_csv(gz, header=None, names=cols, dtype='float32')
    print("HIGGS full:", full.shape)   # kỳ vọng (11000000, 29)
    frac = N_SAMPLE / len(full)
    sample = (full.groupby('target', group_keys=False)
                  .sample(frac=frac, random_state=seed)
                  .reset_index(drop=True))
    sample['target'] = sample['target'].astype('int8')
    sample.to_parquet(higgs_path, index=False)
    del full
    print("Đã lưu", higgs_path, sample.shape)

_h = pd.read_parquet(higgs_path)
print(_h.shape)
print(_h['target'].value_counts(normalize=True))

HIGGS mẫu đã có trên Drive, bỏ qua.
(1000000, 29)
target
1    0.52992
0    0.47008
Name: proportion, dtype: float64


## 8. Tạo bản sạch (Giai đoạn 1 — nhóm D)

Đọc từ `data/raw/` (không sửa), ghi ra `data/processed/`. Quyết định xử lý (ghi lại trong `docs/cach-trien-khai-chi-tiet-tung-giai-doan.md`):

- **Beijing:** gộp 12 trạm → bỏ dòng thiếu bất kỳ cột nào (để baseline có Completeness = 1, không lẫn missing tự nhiên với missing tiêm vào) → bỏ cột `No`, `year` → one-hot `wd` và `station`.
- **HIGGS mẫu / Covertype:** không có missing gốc, chỉ kiểm tra rồi lưu lại.

In [10]:
# --- 8.1 Beijing: gộp 12 trạm + tạo bản sạch ---
PROC_DIR = os.path.join(PROJECT_DIR, 'data/processed')
beijing_out = os.path.join(PROC_DIR, 'beijing_clean.parquet')

if os.path.exists(beijing_out):
    print("beijing_clean đã có, bỏ qua.")
else:
    files = sorted(glob.glob(os.path.join(BEIJING_DIR, 'PRSA_Data_*.csv')))
    bj = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)
    print("Sau khi gộp 12 trạm:", bj.shape)                       # kỳ vọng (420768, 18)
    print("Số ô thiếu theo cột:\n", bj.isna().sum()[lambda s: s > 0])

    n0 = len(bj)
    bj = bj.dropna().reset_index(drop=True)
    print(f"Bỏ {n0 - len(bj)} dòng có missing ({(n0 - len(bj)) / n0:.1%}); còn {len(bj)} dòng")

    bj = bj.drop(columns=['No', 'year'])                          # index + năm (không phải feature chất lượng)
    bj = pd.get_dummies(bj, columns=['wd', 'station'], dtype='int8')
    cols = ['PM2.5'] + [c for c in bj.columns if c != 'PM2.5']    # target ở cột đầu
    bj = bj[cols]
    bj.to_parquet(beijing_out, index=False)
    print("Đã lưu", beijing_out, bj.shape)

_b = pd.read_parquet(beijing_out)
print(_b.shape, "| NaN:", int(_b.isna().sum().sum()), "| dòng trùng:", int(_b.duplicated().sum()))
print(_b.columns.tolist())

Sau khi gộp 12 trạm: (420768, 18)
Số ô thiếu theo cột:
 PM2.5     8739
PM10      6449
SO2       9021
NO2      12116
CO       20701
O3       13277
TEMP       398
PRES       393
DEWP       403
RAIN       390
wd        1822
WSPM       318
dtype: int64
Bỏ 38600 dòng có missing (9.2%); còn 382168 dòng
Đã lưu /content/drive/MyDrive/data-quality-project/data/processed/beijing_clean.parquet (382168, 42)
(382168, 42) | NaN: 0 | dòng trùng: 0
['PM2.5', 'month', 'day', 'hour', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd_E', 'wd_ENE', 'wd_ESE', 'wd_N', 'wd_NE', 'wd_NNE', 'wd_NNW', 'wd_NW', 'wd_S', 'wd_SE', 'wd_SSE', 'wd_SSW', 'wd_SW', 'wd_W', 'wd_WNW', 'wd_WSW', 'station_Aotizhongxin', 'station_Changping', 'station_Dingling', 'station_Dongsi', 'station_Guanyuan', 'station_Gucheng', 'station_Huairou', 'station_Nongzhanguan', 'station_Shunyi', 'station_Tiantan', 'station_Wanliu', 'station_Wanshouxigong']


In [11]:
# --- 8.2 HIGGS mẫu + Covertype: kiểm tra rồi lưu bản sạch ---
for name, src, dst in [
    ('higgs_sample', higgs_path, os.path.join(PROC_DIR, 'higgs_sample_clean.parquet')),
    ('covertype',    cover_path, os.path.join(PROC_DIR, 'covertype_clean.parquet')),
]:
    if os.path.exists(dst):
        print(name, "đã có, bỏ qua.")
        continue
    d = pd.read_parquet(src)
    assert d.isna().sum().sum() == 0, f"{name} có missing — cần xử lý"
    print(f"{name}: {d.shape} | dòng trùng: {int(d.duplicated().sum())}")
    d.to_parquet(dst, index=False)       # không sửa nội dung; mẫu HIGGS đã stratified ở bước 7.3
    print("Đã lưu", dst)

for f in sorted(glob.glob(os.path.join(PROC_DIR, '*.parquet'))):
    print(os.path.basename(f), pd.read_parquet(f).shape)

higgs_sample: (1000000, 29) | dòng trùng: 2281
Đã lưu /content/drive/MyDrive/data-quality-project/data/processed/higgs_sample_clean.parquet
covertype: (581012, 55) | dòng trùng: 0
Đã lưu /content/drive/MyDrive/data-quality-project/data/processed/covertype_clean.parquet
beijing_clean.parquet (382168, 42)
covertype_clean.parquet (581012, 55)
higgs_sample_clean.parquet (1000000, 29)


## Ghi chú

- Notebook này chỉ dựng khung môi trường — chưa chạy thí nghiệm thật.
- Theo dõi tiến độ từng bước tại `docs/checklist-trien-khai.md`.
- Giai đoạn 5 (Experiment Runner, 312 lượt) và Giai đoạn 9 (Spark scalability) nên chạy trên Colab như đã trao đổi; Giai đoạn 2-4 (viết module) có thể làm local trong VS Code rồi đồng bộ code lên Drive để chạy thử ở đây.